In [ ]:
# 공통 경로와 패키지
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [ ]:
table_root = str(PROCESSED_DIR)

table_names = ['AEGIS-Order-Med1',
 'AEGIS-Basic-Order',
 'AEGIS-Weight-and-Height',
 'AEGIS-Nursing-Initial-Assessment',
 'AEGIS-Dx-KCD',
 'AEGIS-Order-CIV2',
 'AEGIS-Order-Test',
 'AEGIS-Order-CIV1',
 'AEGIS-Administer-Med',
 'AEGIS-Flowsheet-Value',
 'AEGIS-Basic-Info',
 'AEGIS-Dx-SNOMED']

all_patient_ids = set()

for folder in table_names:
    target_path = os.path.join(table_root, folder, 'de')
    csv_files = glob.glob(os.path.join(target_path, '**/*.csv'),recursive = True)

    for file in csv_files:
        try:
            table_df = pd.read_csv(file, usecols = ['pt_id'], dtype = {"pt_id":str} ,sep=",", quotechar ='"', lineterminator = '\n')
            patient_ids = table_df['pt_id'].dropna().unique()
            all_patient_ids.update(patient_ids)

        except ValueError as ve:
            print(f"[WARN] 컬럼 없음: {file}")
        except Exception as e:
            print(f"[ERROR] 오류: {file} -> {e}")
            raise


In [ ]:
# pt_id 노이즈 테이블
patient_ids = sorted(all_patient_ids)
np.random.seed(1)

date_offsets = np.random.choice([-7,7], size = len(patient_ids)) # -7 or +7일
minute_offsets = np.random.randint(-30, 31, size = len(patient_ids)) # -30분 ~ +30분

patient_mapping_df = pd.DataFrame({
    'pt_id': patient_ids,
    'new_pt_id': list(range(1, len(patient_ids)+1)),
    'date_noise': date_offsets,
    'time_noise': minute_offsets
})

patient_mapping_df


In [ ]:
PATIENT_MAPPING_PATH.parent.mkdir(parents=True, exist_ok=True)
patient_mapping_df.to_csv(str(PATIENT_MAPPING_PATH), index = False, encoding = 'utf-8')
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n')
